In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.svm import SVR

# Load Dataset
air_quality_data = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')

# Clean trailing empty columns/rows
air_quality_data = air_quality_data.dropna(how='all').dropna(how='all', axis=1)

# Convert -200 sentinel values to NaN
air_quality_data.replace(-200, np.nan, inplace=True)

# Drop NMHC(GT) due to 90.2% missingness
air_quality_data.drop(columns=['NMHC(GT)'], inplace=True, errors='ignore')

# Drop rows where the target C6H6(GT) is missing (cannot impute ground truth)
air_quality_data.dropna(subset=['C6H6(GT)'], inplace=True)

# Combine Date and Time for chronological ordering
air_quality_data['Datetime'] = pd.to_datetime(air_quality_data['Date'] + ' ' + air_quality_data['Time'], format='%d/%m/%Y %H.%M.%S')
air_quality_data.sort_values('Datetime', inplace=True)
air_quality_data.reset_index(drop=True, inplace=True)

# Separate Target (y) and Features (X)
y = air_quality_data['C6H6(GT)'].values
X = air_quality_data.drop(columns=['C6H6(GT)'])

# 2. Chronological Train-Test Split (80/20) to prevent temporal leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=False
)

print(f"Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")

Train set shape: (7192, 14), Test set shape: (1799, 14)


In [20]:
# Function to extract and encode cyclical hour features
def encode_cyclical_features(X_df):
    X_out = X_df.copy()
    
    # Extract integer hour from Time string
    hours = pd.to_datetime(X_out['Time'], format='%H.%M.%S').dt.hour
    
    # Apply Sine and Cosine Transformations
    X_out['Hour_sin'] = np.sin(2 * np.pi * hours / 24.0)
    X_out['Hour_cos'] = np.cos(2 * np.pi * hours / 24.0)
    
    # Drop non-numeric temporal columns from predictor matrix
    X_out.drop(columns=['Date', 'Time', 'Datetime'], inplace=True, errors='ignore')
    return X_out

# Wrap as scikit-learn FunctionTransformer
cyclical_transformer = FunctionTransformer(encode_cyclical_features)

In [21]:
# Identify numerical feature columns dynamically after cyclical transformation
sample_X = encode_cyclical_features(X_train)
numeric_features = sample_X.columns.tolist()

# Define Preprocessing Pipeline: Median Imputation -> Z-Score Standardization
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# ColumnTransformer applying the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ]
)

# Full Preprocessing Pipeline
full_pipeline = Pipeline(steps=[
    ('cyclical_encoding', cyclical_transformer),
    ('preprocessing', preprocessor)
])

# Fit on training data ONLY and transform both sets
X_train_preprocessed = full_pipeline.fit_transform(X_train)
X_test_preprocessed = full_pipeline.transform(X_test)

In [22]:
# Target Transformation: log(1 + y)
y_train_log = np.log1p(y_train)

# Initialize Models
models = {
    "Linear Regression (OLS)": LinearRegression(),
    "Ridge Regression (L2)": RidgeCV(alphas=np.logspace(-2, 3, 50), cv=5),
    "Support Vector Regression (SVR)": SVR(kernel='rbf', C=10.0, epsilon=0.1)
}

# Dictionary to hold evaluation results
results = []

for name, model in models.items():
    # Fit model on log-transformed target
    model.fit(X_train_preprocessed, y_train_log)
    
    # Predict in log scale
    y_pred_log = model.predict(X_test_preprocessed)
    
    # CRUCIAL: Back-transform predictions to original unit (µg/m³) using exp(y) - 1
    y_pred_original = np.expm1(y_pred_log)
    
    # Calculate performance metrics in real units (µg/m³)
    mae = mean_absolute_error(y_test, y_pred_original)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_original))
    r2 = r2_score(y_test, y_pred_original)
    
    results.append({
        "Model": name,
        "MAE (µg/m³)": round(mae, 4),
        "RMSE (µg/m³)": round(rmse, 4),
        "R² Score": round(r2, 4)
    })

# Summary Table
results_df = pd.DataFrame(results)
print("\n=== Model Performance Evaluation (Real Units) ===")
print(results_df.to_string(index=False))


=== Model Performance Evaluation (Real Units) ===
                          Model  MAE (µg/m³)  RMSE (µg/m³)  R² Score
        Linear Regression (OLS)       0.9684        1.4842    0.9481
          Ridge Regression (L2)       1.0394        1.5639    0.9424
Support Vector Regression (SVR)       0.6231        0.9437    0.9790
